In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter


In [ ]:
pileup = pd.read_csv('CLIP-let7g-gene.pileup', sep='\t', names=['chrom', 'pos', '_ref', 'count', 'basereads', 'quals'])
pileup.tail()

,chrom,pos,_ref,count,basereads,quals
83,chr9,106056122,N,88,<<<<<<<<<C$C$C$C$C$C$C$C$C$C$C$C$C$CCCCCC$C$C$...,BHEIG?DIIEEGIIC;GIHEGBIIIIB1=FII?FEIGGGHDBIG=H...
84,chr9,106056123,N,31,<<<<<<<<<CCCCCCCCCCCCCCCCCCCCCC,BHEIG?DIIIIII>GIGGIGGD>BIHHHIEH
85,chr9,106056124,N,31,<<<<<<<<<AAAAAAAAAAAAAAAAAAAAAA,BHEIG?DIIIIHIGGIGGGIG:9DDBIEGFH
86,chr9,106056125,N,31,<<<<<<<<<GGGGGGGGGGGGGGGGGGGGGG,BHEIG?DIIIIIIGGE@GFIGD;GIGIIFHD
87,chr9,106056126,N,30,<<<<<<<<<GGGGGGGGGGGGGGGGGGGGG,BHEIG?DIIIIGHGHIGHI>G;GGGIGIHG


In [ ]:
toremove = re.compile('[<>$*#^]')
pileup['matches'] = pileup['basereads'].apply(lambda x: toremove.sub('', x))

In [3]:
pileup[['chrom', 'pos', 'matches']]

,chrom,pos,matches
0,chr9,106056039,
1,chr9,106056040,
2,chr9,106056041,
3,chr9,106056042,
4,chr9,106056043,
...,...,...,...
83,chr9,106056122,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC...
84,chr9,106056123,CCCCCCCCCCCCCCCCCCCCCC
85,chr9,106056124,AAAAAAAAAAAAAAAAAAAAAA
86,chr9,106056125,GGGGGGGGGGGGGGGGGGGGGG


In [4]:
pileup[pileup['pos'] == 106056094].iloc[0]['matches']

'GGGGGGAAAAAAAAGGGGGAAAAAAGCCGCAGGATGAGGTGATAAGGGAGGGGTGAAGGGCGGTGAAGGGGAAAAGAGAAAGAAAAATAAAGGGGGAGTGGGAGGAAGAAGAGAATA'

자 이제 데이터 준비가 대충 끝났습니다. 다음 순서로 진행해 보세요. (1-4번 단계는 R을 쓰시는 분들은 R로 보내서 작업하셔도 됩니다.)

1. 각 position별로 base수를 셉니다.
1. 각 position별로 Shannon entropy를 계산합니다.
1. 계산한 Shannon entropy를 [bedgraph format](https://genome.ucsc.edu/goldenPath/help/bedgraph.html)으로 출력합니다. 말은 복잡해도 실제로는 간단합니다. 4칸짜리를 만드시면 됩니다.
1. 결과 파일을 본인의 Google Drive에서 가져옵니다.
1. [UCSC Genome Browser](http://genome.ucsc.edu/cgi-bin/hgTracks?db=mm39&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr9%3A106056039-106056126)에 접속해서 Genome은 mm39로 선택합니다.
1. 그래프 아랫쪽의 add custom tracks 버튼을 누릅니다.
1. Paste URLs or data 옆의 Choose File 버튼을 누르고 우리가 만든 bedgraph 파일을 업로드 합니다.
1. 그리고 이리 저리 감상해 보다가 View 메뉴의 PDF/PS 를 눌러서 인증샷을 한 번 찍습니다.
1. Mirlet7d와 Mirlet7f-1도 한 번 해 봅니다.

In [ ]:

# 1. position별 base 수 세기
def count_bases(seq):
    seq = str(seq).upper()
    c = Counter(seq)
    return pd.Series({
        'A': c.get('A', 0),
        'C': c.get('C', 0),
        'G': c.get('G', 0),
        'T': c.get('T', 0)
    })

base_counts = pileup['matches'].apply(count_bases)

pileup = pd.concat([pileup, base_counts], axis=1)

pileup[['chrom', 'pos', 'count', 'A', 'C', 'G', 'T']].head()

,chrom,pos,count,A,C,G,T
0,chr9,106056039,9,0,0,0,0
1,chr9,106056040,9,0,0,0,0
2,chr9,106056041,9,0,0,0,0
3,chr9,106056042,9,0,0,0,0
4,chr9,106056043,9,0,0,0,0


In [6]:
def shannon_entropy(row):
    counts = np.array([row['A'], row['C'], row['G'], row['T']], dtype=float)
    total = counts.sum()
    
    if total == 0:
        return 0
    
    p = counts[counts > 0] / total
    return -np.sum(p * np.log2(p))

pileup['entropy'] = pileup.apply(shannon_entropy, axis=1)

pileup[['chrom', 'pos', 'count', 'A', 'C', 'G', 'T', 'entropy']].tail()

,chrom,pos,count,A,C,G,T,entropy
83,chr9,106056122,88,0,79,0,0,-0.0
84,chr9,106056123,31,0,22,0,0,-0.0
85,chr9,106056124,31,22,0,0,0,-0.0
86,chr9,106056125,31,0,0,22,0,-0.0
87,chr9,106056126,30,0,0,21,0,-0.0


In [20]:
bedgraph = pd.DataFrame({

    'chrom': pileup['chrom'],
    'start': pileup['pos'] - 1,
    'end': pileup['pos'],
    'entropy': pileup['entropy']

})

with open('CLIP-let7g-entropy-track.bedgraph', 'w') as f:

    f.write(
        'track type=bedGraph '
        'name="CLIP-let7g entropy" '
        'description="CLIP-let7g Shannon entropy" '
        'visibility=full autoScale=on alwaysZero=on '
        'viewLimits=0:0.5 color=0,0,255 graphType=bar '
        'maxHeightPixels=80:40:20\n'
    )

    bedgraph.to_csv(
        f,
        sep='\t',
        header=False,
        index=False
    )

bedgraph.head()

,chrom,start,end,entropy
0,chr9,106056038,106056039,0.0
1,chr9,106056039,106056040,0.0
2,chr9,106056040,106056041,0.0
3,chr9,106056041,106056042,0.0
4,chr9,106056042,106056043,0.0


# 공통 함수만들기

In [10]:
def clean_pileup_bases(x):
    x = str(x)
    
    # read 시작 표시 제거: ^ + mapping quality 한 글자
    x = re.sub(r'\^.', '', x)
    
    # insertion/deletion 정보 제거: +2AG, -2nn 같은 것
    x = re.sub(r'[+-]\d+[ACGTNacgtn]+', '', x)
    
    # pileup 특수문자 제거
    x = re.sub(r'[<>$*#]', '', x)
    
    return x

def count_bases(seq):
    seq = str(seq).upper()
    c = Counter(seq)
    return pd.Series({
        'A': c.get('A', 0),
        'C': c.get('C', 0),
        'G': c.get('G', 0),
        'T': c.get('T', 0)
    })

def shannon_entropy(row):
    counts = np.array([row['A'], row['C'], row['G'], row['T']], dtype=float)
    total = counts.sum()
    
    if total == 0:
        return 0
    
    p = counts[counts > 0] / total
    return -np.sum(p * np.log2(p))

Mirlet7d

In [ ]:
pileup_7d = pd.read_csv(
    'CLIP-Mirlet7d-gene.pileup',
    sep='\t',
    names=['chrom', 'pos', '_ref', 'count', 'basereads', 'quals']
)

pileup_7d['matches'] = pileup_7d['basereads'].apply(clean_pileup_bases)

base_counts_7d = pileup_7d['matches'].apply(count_bases)
pileup_7d = pd.concat([pileup_7d, base_counts_7d], axis=1)

pileup_7d['entropy'] = pileup_7d.apply(shannon_entropy, axis=1)

bedgraph_7d = pd.DataFrame({
    'chrom': pileup_7d['chrom'],
    'start': pileup_7d['pos'] - 1,
    'end': pileup_7d['pos'],
    'entropy': pileup_7d['entropy']
})

with open('CLIP-Mirlet7d-entropy-track.bedgraph', 'w') as f:
    f.write(
        'track type=bedGraph name="CLIP-Mirlet7d entropy" '
        'description="CLIP-Mirlet7d Shannon entropy" '
        'visibility=full autoScale=on alwaysZero=on '
        'viewLimits=0:2 color=0,0,255 graphType=bar '
        'maxHeightPixels=80:40:20\n'
    )
    bedgraph_7d.to_csv(f, sep='\t', header=False, index=False)

pileup_7d[['chrom', 'pos', 'count', 'A', 'C', 'G', 'T', 'entropy']].sort_values(
    'entropy', ascending=False
).head(10)

,chrom,pos,count,A,C,G,T,entropy
41,chr13,48689529,187,0,97,32,58,1.450881
40,chr13,48689528,186,0,131,19,10,0.851251
54,chr13,48689542,106,0,0,1,105,0.077017
43,chr13,48689531,174,0,1,0,173,0.051043
22,chr13,48689510,182,1,181,0,0,0.049157
37,chr13,48689525,186,185,1,0,0,0.048269
32,chr13,48689520,187,186,0,1,0,0.048052
6,chr13,48689494,105,0,0,105,0,-0.000000
7,chr13,48689495,104,0,104,0,0,-0.000000
9,chr13,48689497,104,0,104,0,0,-0.000000


In [ ]:
bedgraph_7d = pd.DataFrame({
    'chrom': pileup_7d['chrom'],
    'start': pileup_7d['pos'] - 1,
    'end': pileup_7d['pos'],
    'entropy': pileup_7d['entropy']
})

with open('CLIP-Mirlet7d-entropy-track.bedgraph', 'w') as f:
    f.write(
        'track type=bedGraph name="CLIP-Mirlet7d entropy" '
        'description="CLIP-Mirlet7d Shannon entropy" '
        'visibility=full autoScale=on alwaysZero=on '
        'viewLimits=0:0.5 color=0,0,255 graphType=bar '
        'maxHeightPixels=80:40:20\n'
    )
    bedgraph_7d.to_csv(f, sep='\t', header=False, index=False)



Mirlet7f-1

In [ ]:
pileup_7f1 = pd.read_csv(
    'CLIP-Mirlet7f-1-gene.pileup',
    sep='\t',
    names=['chrom', 'pos', '_ref', 'count', 'basereads', 'quals']
)

pileup_7f1['matches'] = pileup_7f1['basereads'].apply(clean_pileup_bases)

base_counts_7f1 = pileup_7f1['matches'].apply(count_bases)
pileup_7f1 = pd.concat([pileup_7f1, base_counts_7f1], axis=1)

pileup_7f1['entropy'] = pileup_7f1.apply(shannon_entropy, axis=1)

bedgraph_7f1 = pd.DataFrame({
    'chrom': pileup_7f1['chrom'],
    'start': pileup_7f1['pos'] - 1,
    'end': pileup_7f1['pos'],
    'entropy': pileup_7f1['entropy']
})

with open('CLIP-Mirlet7f-1-entropy-track.bedgraph', 'w') as f:
    f.write(
        'track type=bedGraph name="CLIP-Mirlet7f-1 entropy" '
        'description="CLIP-Mirlet7f-1 Shannon entropy" '
        'visibility=full autoScale=on alwaysZero=on '
        'viewLimits=0:0.5 color=0,0,255 graphType=bar '
        'maxHeightPixels=80:40:20\n'
    )
    bedgraph_7f1.to_csv(f, sep='\t', header=False, index=False)

pileup_7f1[['chrom', 'pos', 'count', 'A', 'C', 'G', 'T', 'entropy']].sort_values(
    'entropy', ascending=False
).head(10)

,chrom,pos,count,A,C,G,T,entropy
33,chr13,48691338,151,0,61,10,80,1.273183
32,chr13,48691337,156,0,132,0,24,0.619382
30,chr13,48691335,156,8,148,0,0,0.291818
31,chr13,48691336,158,0,7,0,151,0.261688
4,chr13,48691309,109,0,0,109,0,-0.000000
5,chr13,48691310,116,0,0,116,0,-0.000000
2,chr13,48691307,109,0,109,0,0,-0.000000
3,chr13,48691308,109,109,0,0,0,-0.000000
8,chr13,48691313,132,132,0,0,0,-0.000000
9,chr13,48691314,131,0,0,131,0,-0.000000


In [ ]:
bedgraph_7f1 = pd.DataFrame({
    'chrom': pileup_7f1['chrom'],
    'start': pileup_7f1['pos'] - 1,
    'end': pileup_7f1['pos'],
    'entropy': pileup_7f1['entropy']
})

with open('CLIP-Mirlet7f-1-entropy-track.bedgraph', 'w') as f:
    f.write(
        'track type=bedGraph name="CLIP-Mirlet7f-1 entropy" '
        'description="CLIP-Mirlet7f-1 Shannon entropy" '
        'visibility=full autoScale=on alwaysZero=on '
        'viewLimits=0:0.5 color=0,0,255 graphType=bar '
        'maxHeightPixels=80:40:20\n'
    )
    bedgraph_7f1.to_csv(f, sep='\t', header=False, index=False)